# Remote↔Central perfSONAR Mesh: Automated Setup & Archiving

This notebook provisions a **central VM** and multiple **remote VMs**, installs perfSONAR components on each, wires up **pSConfig** for test orchestration, and configures **local and optional central archiving** so results appear in Grafana.

## What this notebook does

* **Provisioning**

  * Creates one **central VM** and N **remote VMs**.
  * Installs on **all VMs**: `perfsonar-testpoint`, `perfsonar-archive`, and `perfsonar-grafana`.
* **Configuration**

  * **Central** publishes a **pSConfig** template describing all hosts and archives.
  * **Remote Nodes** subscribe to the central template and also load a **local pSConfig** (self + central).
* **Measurements**

  * Schedules the following tests **in both directions** (A→B and B→A where applicable):

    * **Throughput:** `iperf3` (e.g., `-P 4 -t 60 -i 10 -O 10`)
    * **Latency:** `owping`, `twping`, `halfping`
    * **RTT:** `ping`, `tcpping`
    * **MTU:** `fwmtu`
    * **Clock:** `psclock`
    * **Trace:** `traceroute`
  * Test cadences are parameterized; defaults can be set at the top of the notebook.
* **Archiving**

  * **Local archive on each remote VM** via HTTP archiver → Logstash → OpenSearch.
  * **Optional central archiving** (mirrored submission) for fleet-wide dashboards.
* **Visualization**

  * Each VM runs Grafana with perfSONAR dashboards (OpenSearch or pSConfig-driven views).

## Key parameters (set at the top)

* `TOTAL_NODE_CNT = 3`. Controls the number of nodes/VMs to use for the experiment `1(Central) + (N-1) (Remote)`.
* `TEST_INTERVAL = 10M`. Controls the test interval with allowed values `10M`, `2H`, `4H` and `6H`.
* **Archive behavior:**
  * Always send to **local** archive on each remote server.
  * Optionally **mirror** to central by enabling `USE_CENTRAL_ARCHIVE = True`

![](./images/perfsonar-psconfig.png)

## Import the FABlib Library

FABlib is used to programmatically create and manage FABRIC resources such as slices and VMs.

In [1]:
# ------------------------------------------------------------
# Step 1: Initialize FABlib
# ------------------------------------------------------------
# FABlib is used to create and manage the FABRIC virtual
# machines and networks used in this perfSONAR experiment.
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress, os, json

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
                     
fablib.show_config();

Credential Manager,cm.fabric-testbed.net
Orchestrator,orchestrator.fabric-testbed.net
Project ID,a70de2f5-9e12-4b6b-b412-0ae1a2c553b0
Token File,/home/fabric/.tokens.json
Bastion Host,bastion.fabric-testbed.net
Bastion Username,ranna01_0000554580
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Private Key File,/home/fabric/work/fabric_config/slice_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub
Log File,/tmp/fablib/fablib.log
Log Level,INFO


## Create the Experiment Slice

This step provisions the VMs on the FABRIC testbed.

In [2]:
# ------------------------------------------------------------
# Step 2: Define experiment parameters
# ------------------------------------------------------------
# This experiment uses:
#
#   central node : TACC
#   remote node  : SEAT
#
# The code can support more nodes, but two endpoints are used
# for this study experiment so the results are reproducible.


TOTAL_NODE_CNT = 2

# sites = fablib.get_random_sites(TOTAL_NODE_CNT)
# print(f"Sites: {sites}")

USE_CENTRAL_ARCHIVE = True
TEST_INTERVAL = '10M' # Allowed values 10M, 2H, 4H, 6H

In [3]:
# Fixed FABRIC sites are used so that repeated runs use the
# same network path and can be compared consistently.

sites = ["TACC", "SEAT"]
central_node_name = "central"
remote_node_name = "remote"

In [4]:
# ------------------------------------------------------------
# Step 3: Create the FABRIC slice
# ------------------------------------------------------------
# Create one central VM at TACC and one remote VM at SEAT.
#
# Each node is connected to a FabNetv4 Layer-3 network.
# FABRIC routing allows the two perfSONAR endpoints to
# communicate across the testbed.

slice_name = 'perfSonar-fabric-multi-site'
slice = fablib.get_slice(name=slice_name)

print(type(slice))
print(slice.get_name())
print(slice.get_state())

User: ranna01@jaguar.tamu.edu bastion key is valid!
Configuration is valid
<class 'fabrictestbed_extensions.fablib.slice.Slice'>
perfSonar-fabric-multi-site
StableOK


In [3]:
#Create Slice
slice = fablib.new_slice(name=slice_name)

for i, s in enumerate(sites):
    prefix = central_node_name if i == 0 else remote_node_name

    net = slice.add_l3network(name=f"{s}-l3", type='IPv4')

    node = slice.add_node(
        name=f"{prefix}-{s}",
        site=s,
        image="default_ubuntu_24",
        cores=16,
        ram=32,
        disk=100
    )

    iface = node.add_component(model='NIC_Basic', name='nic1').get_interfaces()[0]
    iface.set_mode('auto')
    net.add_interface(iface)

    node.add_route(subnet=fablib.FABNETV4_SUBNET, next_hop=net.get_gateway())



#step4: verify the Slice Request
slice.submit();


Retry: 15, Time: 338 sec


ID,2ee43d85-e88f-4005-9b78-2279dec22370
Name,perfSonar-fabric-multi-site
Lease Expiration (UTC),2026-09-17 01:49:49 +0000
Lease Start (UTC),2026-09-16 01:49:49 +0000
Project ID,a70de2f5-9e12-4b6b-b412-0ae1a2c553b0
State,StableOK
Email,ranna01@jaguar.tamu.edu
UserId,e335f96a-332c-49ec-9810-0d6cbe323c83


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
07bce9dc-9c33-4234-a557-040f61182261,central-TACC,16,32,100,default_ubuntu_24,qcow2,tacc-w2.fabric-testbed.net,TACC,ubuntu,2605:2800:2011:201:f816:3eff:fe57:3678,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2605:2800:2011:201:f816:3eff:fe57:3678,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
cfd8a42a-9365-4df9-b016-969a25bbb818,remote-SEAT,16,32,100,default_ubuntu_24,qcow2,seat-w1.fabric-testbed.net,SEAT,ubuntu,2001:400:a100:3080:f816:3eff:fe12:6c44,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3080:f816:3eff:fe12:6c44,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key


ID,Name,Layer,Type,Site,Gateway,Subnet,State,Error
dd4d1d6f-a931-4a5f-8d4c-cfbc8a557812,SEAT-l3,L3,FABNetv4,SEAT,10.139.9.1,10.139.9.0/24,Active,
f5169324-b97b-4d73-842d-6b6936ae9752,TACC-l3,L3,FABNetv4,TACC,10.130.142.1,10.130.142.0/24,Active,


Name,Short Name,Node,Network,Bandwidth,VLAN,MAC,Physical Device,Device,Mode,IP Address,Numa Node,Switch Port
central-TACC-nic1-p1,p1,central-TACC,TACC-l3,100,,2A:44:B1:FC:07:89,enp7s0,enp7s0,auto,10.130.142.2,6,HundredGigE0/0/0/7
remote-SEAT-nic1-p1,p1,remote-SEAT,SEAT-l3,100,,16:20:CB:2D:22:69,enp7s0,enp7s0,auto,10.139.9.2,4,HundredGigE0/0/0/5



Time to print interfaces 339 seconds


## Configure `/etc/hosts`

For consistent name resolution across the fleet, update `/etc/hosts` on **every** VM with the same mappings.

**Steps**

* Add an entry for each node: IP address followed by its hostname (and optional aliases).
* Keep the file identical on all nodes to prevent lookup mismatches during monitoring/orchestration.

**Example**

```
10.128.1.2    central-STAR
10.128.127.2  remote-RUTG
10.128.254.2  remote-NEWY
```

## Clone the repository

In [5]:
slice = fablib.get_slice(slice_name)

for server in slice.get_nodes():
    stdout, stderr = server.execute("git clone https://github.com/kthare10/perfsonar-extensions.git /home/ubuntu/perfsonar-extensions")

Cloning into '/home/ubuntu/perfsonar-extensions'...
Cloning into '/home/ubuntu/perfsonar-extensions'...


## Install perfSONAR stack (Testpoint, Archive, Grafana)

Install the full perfSONAR stack on **every** VM—both central and remote—for a consistent setup:

* **Testpoint**: pScheduler and measurement tools.
* **Archive**: Logstash/OpenSearch pipeline for storing results.
* **Grafana**: Dashboards for visualization.

This ensures each node can run tests, store results locally, and (optionally) mirror data to the central archive.

In [6]:
ip_map = {}
central_host = ""
for n in slice.get_nodes():
    ip_addr = n.get_interface(network_name=f"{n.get_site()}-l3").get_ip_addr() 
    ip_map[n.get_name()] = ip_addr
    if central_node_name in n.get_name():
        central_host = n.get_name()
    n.execute('cd perfsonar-extensions/native/ && sudo scripts/perfsonar-install.sh', quiet=True, output_file=f"{n.get_name()}.log")

In [10]:
for n in slice.get_nodes():
    print("\n==============================")
    print(n.get_name())
    print("==============================")

    stdout, stderr = n.execute("""
        echo "pscheduler:"
        command -v pscheduler || echo "MISSING"

        echo "psconfig:"
        command -v psconfig || echo "MISSING"

        echo "logstash apache config:"
        if [ -f /etc/apache2/conf-available/apache-logstash.conf ]; then
            echo "FOUND"
        else
            echo "MISSING"
        fi
    """)


central-TACC
pscheduler:
/usr/bin/pscheduler
psconfig:
/usr/bin/psconfig
logstash apache config:
FOUND

remote-SEAT
pscheduler:
/usr/bin/pscheduler
psconfig:
/usr/bin/psconfig
logstash apache config:
FOUND


In [11]:
for n in slice.get_nodes():
    logfile = f"{n.get_name()}.log"
    print("\n==========", logfile, "==========")
    with open(logfile, "r", errors="ignore") as f:
        print("".join(f.readlines()[-30:]))


========== central-TACC.log ==========
apache2_invoke ssl: already enabled
apache2_invoke proxy: already enabled
apache2_invoke proxy_http: already enabled
Setting up perfsonar-grafana-toolkit (5.2.6-1) ...
Processing triggers for man-db (2.12.0-4build2) ...
debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype

Running kernel seems to be up-to-date.

Restarting services...

Service restarts being deferred:
 /etc/needrestart/restart.d/dbus.service
 systemctl restart getty@tty1.service
 systemctl restart serial-getty@ttyS0.service
 systemctl restart systemd-logind.service
 systemctl restart unattended-upgrades.service

No containers need to be restarted.

User sessions running outdated bin

## Archive access control — update Apache `/logstash` allowlist & auth

Lock down the Logstash ingest endpoint so only approved hosts (and/or authenticated clients) can post results.

**Steps**

1. **Edit the `/logstash` location block**
   File: `/etc/apache2/conf-available/apache-logstash.conf`

```apache
# ... existing ProxyPass/ProxyPassReverse for /logstash ...

<Location "/logstash">
  AuthType Basic
  AuthName "Logstash Ingest"
  # User file created by the archive install (adjust if different)
  AuthUserFile /etc/perfsonar/opensearch/logstash_login

  # Allow either a valid user OR specific IPs (including localhost)
  <RequireAny>
    Require valid-user
    Require ip 127.0.0.1 ::1
    # Add remote testpoints here (examples):
    Require ip 10.128.127.0/24
    Require ip 203.0.113.45
  </RequireAny>
</Location>
```

3. **Validate & reload Apache**

```bash
sudo apachectl -t
sudo systemctl reload apache2
```

4. **Quick checks**

* On the archive host: `sudo tail -f /var/log/apache2/access.log /var/log/apache2/error.log`
* From a remote VM, run a test that uses the HTTP archiver, then on the archive:

  ```bash
  pscheduler archiving-summary PT30M
  ```

  You should see successes increase.

**Notes**

* Keep HTTPS enabled for `/logstash` and rotate credentials as needed.
* Using both `Require valid-user` and `Require ip` lets you mix “trusted by IP” remote VMs with authenticated clients.

In [12]:
all_ips = ip_map.values()
for n in slice.get_nodes():
    n.execute(f'cd perfsonar-extensions/native/ && sudo scripts/allow_logstash_ips.sh  {" ".join(map(str, all_ips))}')

Backing up /etc/apache2/conf-available/apache-logstash.conf -> /etc/apache2/conf-available/apache-logstash.conf.bak.20260916-034132
Adding: Require ip 10.130.142.2
Adding: Require ip 10.139.9.2
Validating Apache config...
[Wed Sep 16 03:41:32.835967 2026] [proxy:warn] [pid 52391:tid 135733884036992] AH01146: Ignoring parameter 'status=+I' for worker 'http://localhost:3000' because of worker sharing
Syntax OK
Reloading Apache...
Done. Updated /etc/apache2/conf-available/apache-logstash.conf
Backing up /etc/apache2/conf-available/apache-logstash.conf -> /etc/apache2/conf-available/apache-logstash.conf.bak.20260916-034133
Adding: Require ip 10.130.142.2
Adding: Require ip 10.139.9.2
Validating Apache config...
[Wed Sep 16 03:41:33.190548 2026] [proxy:warn] [pid 52023:tid 135595168184192] AH01146: Ignoring parameter 'status=+I' for worker 'http://localhost:3000' because of worker sharing
Syntax OK
Reloading Apache...
Done. Updated /etc/apache2/conf-available/apache-logstash.conf


In [13]:
all_hosts = [x for name, ip in ip_map.items() for x in (name, str(ip))]
print(all_hosts)

['central-TACC', '10.130.142.2', 'remote-SEAT', '10.139.9.2']


In [14]:
print(" ".join(map(str, all_hosts)))

central-TACC 10.130.142.2 remote-SEAT 10.139.9.2


In [15]:
for n in slice.get_nodes():
    n.upload_directory('psconfig', '.')

## pSConfig

Orchestrate tests with a central template and have remote VMs subscribe to it.

### Central VM: build & publish the mesh template

1. Create a JSON template defining **addresses, groups, tests, schedules, and archives** (the HTTP `/logstash` archive) using `psconfig/psconfig_builder.py`.
2. Publish it so remote VMs can subscribe:

   ```bash
   sudo psconfig publish psconfig/psconfig.json
   sudo psconfig remote add https://localhost/psconfig/psconfig.json
   ```

### Remote VMs: subscribe (and auto-configure archives)

1. Create a JSON template defining **addresses, groups, tests, schedules, and archives** (the HTTP `/logstash` archive) using `psconfig/psconfig_builder.py`.
2. Add a small **local** template on each remote VM (self + central) for specific extras, and subscribe to it as a file:

   ```bash
   sudo psconfig publish psconfig/psconfig.json
   sudo psconfig remote add https://localhost/psconfig/psconfig.json
   ```

### Verify

* After \~1 minute, tasks should appear:

  ```bash
  pscheduler schedule
  pscheduler archiving-summary PT30M
  ```
* In Grafana (central and/or remote), confirm data is arriving for the time window covering the first runs.


In [16]:
for n in slice.get_nodes():
    if central_host in n.get_name():
        cmd = (
                f'cd perfsonar-extensions/native/ && python3 psconfig/psconfig_builder.py --base_config_file psconfig/base_psconfig.json '
                f'--output_file psconfig/psconfig.json --no_add_tests --host_list {" ".join(map(str, all_hosts))}'
        )
        print(cmd)
        stdout, stderr = n.execute(cmd)
    else:
        if USE_CENTRAL_ARCHIVE:
            cmd = (
                    f"cd perfsonar-extensions/native/ && python3 psconfig/psconfig_builder.py "
                    f"--base_config_file psconfig/base_psconfig.json "
                    f"--output_file psconfig/psconfig.json "
                    f"--remote {ip_map[central_host]} "
                    f'--host_list {n.get_name()} {ip_map[n.get_name()]} {central_host} {ip_map[central_host]}'
                )
        else:
            cmd = (
                    f"cd perfsonar-extensions/native/ && python3 psconfig/psconfig_builder.py "
                    f"--base_config_file psconfig/base_psconfig.json "
                    f"--output_file psconfig/psconfig.json "
                    f'--host_list {n.get_name()} {ip_map[n.get_name()]} {central_host} {ip_map[central_host]}'
                )
            
        print(cmd)
        stdout, stderr = n.execute(cmd)

    stdout, stderr = n.execute('cd perfsonar-extensions/native/ && sudo psconfig validate psconfig/psconfig.json')
    stdout, stderr = n.execute('cd perfsonar-extensions/native/ && sudo psconfig publish psconfig/psconfig.json')

    stdout, stderr = n.execute('sudo psconfig remote add "https://localhost/psconfig/psconfig.json"')
    print("==============================================================================================================================================")
    print()

cd perfsonar-extensions/native/ && python3 psconfig/psconfig_builder.py --base_config_file psconfig/base_psconfig.json --output_file psconfig/psconfig.json --no_add_tests --host_list central-TACC 10.130.142.2 remote-SEAT 10.139.9.2
/usr/lib/python3/dist-packages/psconfig/client/psconfig/parsers/task_generator.py:244: SyntaxWarning: invalid escape sequence '\d'
  test_data_json = re.sub("['\"](\d+)['\"]", r'\1',test_data_json) #removes both single and double quotes
Loading config ...... OK
Validating JSON schema ...... OK
Verifying object references ...... OK
pScheduler Validation (Quick) ...... OK

pSConfig JSON is valid
                                               
Success! File saved to /usr/lib/perfsonar/web-psconfig/psconfig.json
Published file can be accessed at https://central-TACC/psconfig/psconfig.json
Execute the following on a host running an agent to use this file:
    psconfig remote add "https://central-TACC/psconfig/psconfig.json"

=== pScheduler Agent ===
Added remote 

In [6]:
"""before Grafana, run these verification commands on both nodes:"""
for n in slice.get_nodes():
    print("\n==============================")
    print(n.get_name())
    print("==============================")

    n.execute("""
        echo "=== pScheduler schedule ==="
        pscheduler schedule

        echo
        echo "=== Archiving summary ==="
        pscheduler archiving-summary PT30M
    """)


central-TACC
=== pScheduler schedule ===
2026-10-18T03:42:44+00:00 - 2026-10-19T03:42:46+00:00 (Pending)
latencybg --source 10.139.9.2 --dest 10.130.142.2 --flip (Run with tool 'powstream')
https://central-tacc/pscheduler/tasks/b2b2612a-303b-4191-b3e4-f8e83665a28d/runs/4f011d42-a1f8-4b05-88d6-e4cd13b748c5


2026-10-18T04:18:32+00:00 - 2026-10-19T04:18:34+00:00 (Pending)
latencybg --source 10.139.9.2 --dest 10.130.142.2 --flip (Run with tool 'powstream')
https://central-tacc/pscheduler/tasks/e44411cc-fa0f-4ff5-b81b-cfd5ba1f900e/runs/1e7222ec-2f9f-489b-bd48-c09643726514


2026-10-18T05:18:56+00:00 - 2026-10-19T05:18:58+00:00 (Pending)
latencybg --source 10.139.9.2 --dest 10.130.142.2 --flip (Run with tool 'powstream')
https://central-tacc/pscheduler/tasks/b80b6cb7-e16d-4ea6-a5ce-5030062831ae/runs/c223c4aa-560a-4723-90d0-8bea89897bf4


2026-10-18T06:19:06+00:00 - 2026-10-19T06:19:08+00:00 (Pending)
latencybg --source 10.139.9.2 --dest 10.130.142.2 --flip (Run with tool 'powstream')
https

In [5]:
for n in slice.get_nodes():
    print("\n==============================")
    print(n.get_name())
    print("==============================")
    n.execute("""
        echo "=== Archiving summary ==="
        pscheduler archiving-summary PT1H
    """)


central-TACC
=== Archiving summary ===
Nothing archived since 2026-10-19 02:33:59.919627+00:00

remote-SEAT
=== Archiving summary ===
Nothing archived since 2026-10-19 02:34:02.168236+00:00


## Create SSH Tunnel (Grafana Access)

Use an SSH local port forward so you can open the remote Grafana UI in your local browser.

In [19]:
fablib.create_ssh_tunnel_config(overwrite=True)


SSH tunnel config created and zipped at: /home/fabric/work/fabric_config/fabric_ssh_tunnel_tools.tgz

Download Instructions:
Download your custom `fabric_ssh_tunnel_tools.tgz` file from the `fabric_config` folder.

Usage Instructions:
1. Unzip the archive and place the resulting `fabric_ssh_tunnel_tools/` folder somewhere accessible from your terminal.
2. Open a terminal window (on Windows, use PowerShell).
3. Use `cd` to navigate into the `fabric_ssh_tunnel_tools` folder.
4. In your terminal, run the SSH tunnel command generated by the next notebook cell.
    


## perfSONAR Toolkit Grafana

Use SSH local port forwards to map your **local** ports to each remote host’s **HTTPS (443)**. Then open the URLs below in your browser.

### Central Node (maps local 8443 → remote 443)

```bash
ssh -N -L 8443:localhost:443 <USER>@<CENTRAL_IP>
# Browse: https://127.0.0.1:8443
```

### Remote VMs (add one tunnel per VM)

```bash
# Remote-1  (local 8444 → remote-1:443)
ssh -N -L 8444:localhost:443 <USER>@<REMOTE1_IP>
# Browse: https://127.0.0.1:8444

# Remote-2  (local 8445 → remote-2:443)
ssh -N -L 8445:localhost:443 <USER>@<REMOTE2_IP>
# Browse: https://127.0.0.1:8445
```

> Tip: add `-f` to send the tunnel to the background (`ssh -fN -L ...`).
> If remote VMs are only reachable via central, use a jump host:
> `ssh -J <USER>@<CENTRAL_IP> -N -L 8444:localhost:443 <USER>@REMOTE1_IP>`

### Notes

* perfSONAR/Grafana often uses a self-signed cert — your browser may warn; proceed if you trust the host.
* Log in with your Grafana credentials.
* Stop a tunnel with `Ctrl+C` (or kill the background `ssh` if you used `-f`).


In [5]:
# Port on your local machine that you want to map the File Browser to.
# local_port=8443
local_port=8453
# Local interface to map the File Browser to (can be `localhost`)
local_host='127.0.0.1'

# Port on the node used by the File Browser Service
target_port='443'

for n in slice.get_nodes():
    # Username/node on FABRIC
    target_host=f'{n.get_username()}@{n.get_management_ip()}'
    print(f'Tunnel command for {n.get_name()}')
    print(f'ssh  -L {local_host}:{local_port}:127.0.0.1:{target_port} -i {os.path.basename(fablib.get_default_slice_public_key_file())[:-4]} -F ssh_config {target_host}')
    local_port += 1
    print()


Tunnel command for central-TACC
ssh  -L 127.0.0.1:8453:127.0.0.1:443 -i slice_key -F ssh_config ubuntu@2605:2800:2011:201:f816:3eff:fe57:3678

Tunnel command for remote-SEAT
ssh  -L 127.0.0.1:8454:127.0.0.1:443 -i slice_key -F ssh_config ubuntu@2001:400:a100:3080:f816:3eff:fe12:6c44



## Verification

### pScheduler health

```bash
# Basic end-to-end checks (network, tools, auth, archives)
pscheduler troubleshoot

# What’s scheduled / running
pscheduler schedule
pscheduler task --state on-run

# Archiving success/fail in the last 30 minutes
pscheduler archiving-summary PT30M
```

### OpenSearch spot-checks (recent docs)

```bash
# Replace with your archive host and credentials
curl -k -u '<user>:<pass>' \
  'https://<archive-host>/opensearch/pscheduler*/_search?size=5' \
  -H 'Content-Type: application/json' -d '{
    "sort": [{"pscheduler.start_time":{"order":"desc"}}]
  }' | jq '.[].hits.hits[]._source
           | { time: .pscheduler.start_time,
               type: .test.type,
               src: .test.spec.source,
               dst: .test.spec.dest,
               summary: .summary }'
```

You should see `src` and `dst` populated; if blank, ensure tests specify **both** `--source` and `--dest` (pSConfig does this automatically).

### Grafana dashboards

* Open the dashboard and set the time range to include your latest runs.
* Confirm panels populate and show **source/dest** labels.
* If no data:

  * Check the Grafana data source (OpenSearch URL, index `pscheduler*`, time field `pscheduler.start_time`).
  * Re-check `pscheduler archiving-summary` and the archive’s Apache/Logstash logs.


## Delete the Slice

Please delete your slice when you are done with your experiment.

In [ ]:
# slice = fablib.get_slice(slice_name)
# slice.delete()

In [8]:
central = slice.get_node(name="central-TACC")

stdout, stderr = central.execute(r"""
echo "=== OPENSEARCH CONFIG CHECK ==="

echo
echo "Login file exists?"
sudo test -f /etc/perfsonar/opensearch/opensearch_login \
  && echo "YES" \
  || echo "NO"

echo
echo "Login file format:"
sudo awk -F: '
{
  if (NF >= 2)
    print "username:password format"
  else
    print "single-value/other format"
}' /etc/perfsonar/opensearch/opensearch_login 2>/dev/null

echo
echo "Grafana datasource references:"
sudo grep -RniE "opensearch|pscheduler" \
  /etc/grafana/provisioning/datasources \
  2>/dev/null | head -20

echo
echo "OpenSearch service status:"
sudo systemctl is-active opensearch 2>/dev/null || true
""")

print(stdout)

if stderr:
    print(stderr)

=== OPENSEARCH CONFIG CHECK ===

Login file exists?
YES

Login file format:
single-value/other format

Grafana datasource references:

OpenSearch service status:
active
=== OPENSEARCH CONFIG CHECK ===

Login file exists?
YES

Login file format:
single-value/other format

Grafana datasource references:

OpenSearch service status:
active



In [10]:
central = slice.get_node(name="central-TACC")

stdout, stderr = central.execute(r"""
echo "=== OPENSEARCH AUTH TEST ==="

sudo bash -c '
read -r OS_USER OS_PASS < /etc/perfsonar/opensearch/opensearch_login

CODE=$(curl -ks \
    --connect-timeout 5 \
    --max-time 10 \
    -o /dev/null \
    -w "%{http_code}" \
    -u "${OS_USER}:${OS_PASS}" \
    https://localhost/opensearch/pscheduler/_search)

echo "OpenSearch HTTP status: $CODE"
'
""")

print(stdout)

if stderr:
    print(stderr)

=== OPENSEARCH AUTH TEST ===
OpenSearch HTTP status: 200
=== OPENSEARCH AUTH TEST ===
OpenSearch HTTP status: 200



# Automated Archive Analysis Extensions
This section analyzes the complete OpenSearch archive for the SEAT→TACC experiment and adds quality scoring, archive-derived degradation detection, and an automated experiment summary.


In [13]:
# ============================================================
# NOTEBOOK 2 - Safe archive count by test type
# Avoids aggregation on the text field test.type
# ============================================================

import json

central = slice.get_node(name="central-TACC")

test_types = [
    "throughput",
    "rtt",
    "latencybg",
    "mtu",
    "trace",
    "clock"
]

print("=== NOTEBOOK 2 ARCHIVE SUMMARY ===")
print()

# ------------------------------------------------------------
# Total archive document count
# ------------------------------------------------------------

total_cmd = r'''
sudo bash -c '
read -r OS_USER OS_PASS < /etc/perfsonar/opensearch/opensearch_login

curl -ks \
  -u "${OS_USER}:${OS_PASS}" \
  -H "Content-Type: application/json" \
  https://localhost/opensearch/pscheduler/_search \
  -d '"'"'{
        "size": 0,
        "track_total_hits": true,
        "query": {
          "match_all": {}
        }
      }'"'"'
'
'''

stdout, stderr = central.execute(total_cmd)

data = json.loads(stdout)

total = data["hits"]["total"]["value"]

print(f"Total archived documents: {total}")
print()


# ------------------------------------------------------------
# Count each test type separately
# ------------------------------------------------------------

for test_type in test_types:

    cmd = f'''
sudo bash -c '
read -r OS_USER OS_PASS < /etc/perfsonar/opensearch/opensearch_login

curl -ks \
  -u "${{OS_USER}}:${{OS_PASS}}" \
  -H "Content-Type: application/json" \
  https://localhost/opensearch/pscheduler/_search \
  -d '"'"'{{
        "size": 0,
        "track_total_hits": true,
        "query": {{
          "match": {{
            "test.type": "{test_type}"
          }}
        }}
      }}'"'"'
'
'''

    stdout, stderr = central.execute(cmd)

    result = json.loads(stdout)

    count = result["hits"]["total"]["value"]

    print(f"{test_type:<15} {count} records")

=== NOTEBOOK 2 ARCHIVE SUMMARY ===

{"took":5,"timed_out":false,"_shards":{"total":30,"successful":30,"skipped":0,"failed":0},"hits":{"total":{"value":927599,"relation":"eq"},"max_score":null,"hits":[]}}Total archived documents: 927599

{"took":5,"timed_out":false,"_shards":{"total":30,"successful":30,"skipped":0,"failed":0},"hits":{"total":{"value":2770,"relation":"eq"},"max_score":null,"hits":[]}}throughput      2770 records
{"took":4,"timed_out":false,"_shards":{"total":30,"successful":30,"skipped":0,"failed":0},"hits":{"total":{"value":3182,"relation":"eq"},"max_score":null,"hits":[]}}rtt             3182 records
{"took":5,"timed_out":false,"_shards":{"total":30,"successful":30,"skipped":0,"failed":0},"hits":{"total":{"value":912105,"relation":"eq"},"max_score":null,"hits":[]}}latencybg       912105 records
{"took":7,"timed_out":false,"_shards":{"total":30,"successful":30,"skipped":0,"failed":0},"hits":{"total":{"value":3181,"relation":"eq"},"max_score":null,"hits":[]}}mtu         

In [7]:
# ============================================================
# CONTRIBUTION TASK 1
# Automatic Full-Archive Site Performance Analysis
# Notebook 2: remote-SEAT -> central-TACC
# ============================================================

import json
import base64

central = slice.get_node(name="central-TACC")


def query_opensearch(query_body):

    # Convert Python dictionary -> JSON -> Base64
    body_json = json.dumps(query_body)
    body_b64 = base64.b64encode(
        body_json.encode("utf-8")
    ).decode("utf-8")

    cmd = f"""
sudo bash -c '
read -r OS_USER OS_PASS < /etc/perfsonar/opensearch/opensearch_login

echo "{body_b64}" | base64 -d | \
curl -ks \
  -u "${{OS_USER}}:${{OS_PASS}}" \
  -H "Content-Type: application/json" \
  --data-binary @- \
  https://localhost/opensearch/pscheduler/_search
'
"""

    stdout, stderr = central.execute(cmd)

    data = json.loads(stdout)

    # Stop clearly if OpenSearch returns an error
    if "error" in data:
        raise RuntimeError(
            "OpenSearch error: " +
            json.dumps(data["error"], indent=2)
        )

    return data


# ============================================================
# 1. THROUGHPUT
# ============================================================

throughput_query = {
    "size": 0,
    "track_total_hits": True,
    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "throughput"}},
                {"term": {"result.succeeded": True}}
            ]
        }
    },
    "aggs": {
        "avg_value": {
            "avg": {"field": "result.throughput"}
        },
        "min_value": {
            "min": {"field": "result.throughput"}
        },
        "max_value": {
            "max": {"field": "result.throughput"}
        }
    }
}

t_data = query_opensearch(throughput_query)

throughput_count = t_data["hits"]["total"]["value"]

throughput_avg = (
    t_data["aggregations"]["avg_value"]["value"]
    / 1_000_000
)

throughput_min = (
    t_data["aggregations"]["min_value"]["value"]
    / 1_000_000
)

throughput_max = (
    t_data["aggregations"]["max_value"]["value"]
    / 1_000_000
)


# ============================================================
# 2. RTT + PACKET LOSS
# ============================================================

rtt_query = {
    "size": 0,
    "track_total_hits": True,
    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "rtt"}},
                {"term": {"result.succeeded": True}}
            ]
        }
    },
    "aggs": {
        "avg_rtt": {
            "avg": {"field": "result.rtt.mean"}
        },
        "min_rtt": {
            "min": {"field": "result.rtt.mean"}
        },
        "max_rtt": {
            "max": {"field": "result.rtt.mean"}
        },
        "avg_loss": {
            "avg": {"field": "result.packets.loss"}
        }
    }
}

r_data = query_opensearch(rtt_query)

rtt_count = r_data["hits"]["total"]["value"]

# RTT stored in seconds -> milliseconds
rtt_avg = (
    r_data["aggregations"]["avg_rtt"]["value"]
    * 1000
)

rtt_min = (
    r_data["aggregations"]["min_rtt"]["value"]
    * 1000
)

rtt_max = (
    r_data["aggregations"]["max_rtt"]["value"]
    * 1000
)

# Loss stored as fraction -> percentage
rtt_loss = (
    r_data["aggregations"]["avg_loss"]["value"]
    * 100
)


# ============================================================
# 3. LATENCY + JITTER + PACKET LOSS
# ============================================================

latency_query = {
    "size": 0,
    "track_total_hits": True,
    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "latencybg"}},
                {"term": {"result.succeeded": True}}
            ]
        }
    },
    "aggs": {
        "avg_latency": {
            "avg": {"field": "result.latency.mean"}
        },
        "avg_jitter": {
            "avg": {"field": "result.latency.stddev"}
        },
        "avg_loss": {
            "avg": {"field": "result.packets.loss"}
        }
    }
}

l_data = query_opensearch(latency_query)

latency_count = l_data["hits"]["total"]["value"]

latency_avg = (
    l_data["aggregations"]["avg_latency"]["value"]
)

jitter_avg = (
    l_data["aggregations"]["avg_jitter"]["value"]
)

latency_loss = (
    l_data["aggregations"]["avg_loss"]["value"]
    * 100
)


# ============================================================
# CLEAN OUTPUT
# ============================================================

print()
print("=" * 62)
print("FULL ARCHIVE AUTOMATIC SITE PERFORMANCE ANALYSIS")
print("Notebook 2: remote-SEAT -> central-TACC")
print("=" * 62)

print()
print(f"Throughput records : {throughput_count}")
print(f"Average Throughput : {throughput_avg:.3f} Mbps")
print(f"Minimum Throughput : {throughput_min:.3f} Mbps")
print(f"Maximum Throughput : {throughput_max:.3f} Mbps")

print()
print(f"RTT records        : {rtt_count}")
print(f"Average RTT        : {rtt_avg:.3f} ms")
print(f"Minimum RTT        : {rtt_min:.3f} ms")
print(f"Maximum RTT        : {rtt_max:.3f} ms")
print(f"RTT Packet Loss    : {rtt_loss:.3f}%")

print()
print(f"Latency records    : {latency_count}")
print(f"Average Latency    : {latency_avg:.3f} ms")
print(f"Average Jitter     : {jitter_avg:.3f} ms")
print(f"Latency Packet Loss: {latency_loss:.3f}%")

print()
print("Task 1 completed successfully.")

{"took":5045,"timed_out":false,"_shards":{"total":60,"successful":60,"skipped":0,"failed":0},"hits":{"total":{"value":7042,"relation":"eq"},"max_score":null,"hits":[]},"aggregations":{"min_value":{"value":3820547.0},"avg_value":{"value":3.2416532936474013E8},"max_value":{"value":3.510624E8}}}{"took":43,"timed_out":false,"_shards":{"total":60,"successful":60,"skipped":0,"failed":0},"hits":{"total":{"value":8044,"relation":"eq"},"max_score":null,"hits":[]},"aggregations":{"avg_loss":{"value":7.458975745160327E-5},"max_rtt":{"value":0.23424799740314484},"avg_rtt":{"value":0.09333430557761366},"min_rtt":{"value":0.09239699691534042}}}{"took":504,"timed_out":false,"_shards":{"total":60,"successful":60,"skipped":0,"failed":0},"hits":{"total":{"value":2218616,"relation":"eq"},"max_score":null,"hits":[]},"aggregations":{"avg_loss":{"value":6.381345525650351E-5},"avg_latency":{"value":42.683122358167914},"avg_jitter":{"value":0.27031441935944955}}}
FULL ARCHIVE AUTOMATIC SITE PERFORMANCE ANALYS

In [8]:
# ============================================================
# CONTRIBUTION TASK 2
# Automatic Site Quality Score
# Notebook 2: remote-SEAT -> central-TACC
# ============================================================

# Project-defined reference thresholds
# These are heuristic values for this research project,
# not official perfSONAR standards.

THROUGHPUT_TARGET = 500.0   # Mbps
RTT_LIMIT = 200.0           # ms
LOSS_LIMIT = 5.0            # %
JITTER_LIMIT = 100.0        # ms


# ------------------------------------------------------------
# Normalize metrics to 0-100
# ------------------------------------------------------------

# Higher throughput is better
throughput_score = min(
    100,
    (throughput_avg / THROUGHPUT_TARGET) * 100
)

# Lower RTT is better
rtt_score = max(
    0,
    100 - (rtt_avg / RTT_LIMIT) * 100
)

# Lower packet loss is better
loss_score = max(
    0,
    100 - (latency_loss / LOSS_LIMIT) * 100
)

# Lower jitter is better
jitter_score = max(
    0,
    100 - (jitter_avg / JITTER_LIMIT) * 100
)


# ------------------------------------------------------------
# Weighted quality score
# ------------------------------------------------------------

quality_score = (
    throughput_score * 0.40 +
    rtt_score        * 0.30 +
    loss_score       * 0.20 +
    jitter_score     * 0.10
)


# ------------------------------------------------------------
# Quality classification
# ------------------------------------------------------------

if quality_score >= 90:
    quality_level = "Excellent"

elif quality_score >= 75:
    quality_level = "Good"

elif quality_score >= 60:
    quality_level = "Moderate"

else:
    quality_level = "Poor"


# ------------------------------------------------------------
# Display result
# ------------------------------------------------------------

print("=" * 62)
print("Automatic Site-to-Site Path Quality Score")
print("Notebook 2: remote-SEAT -> central-TACC")
print("=" * 62)

print()
print("Historical Average Metrics")
print("--------------------------")

print(f"Throughput       : {throughput_avg:.3f} Mbps")
print(f"RTT              : {rtt_avg:.3f} ms")
print(f"Packet Loss      : {latency_loss:.3f}%")
print(f"Jitter           : {jitter_avg:.3f} ms")

print()
print("Normalized Metric Scores")
print("------------------------")

print(f"Throughput Score : {throughput_score:.2f}/100")
print(f"RTT Score        : {rtt_score:.2f}/100")
print(f"Loss Score       : {loss_score:.2f}/100")
print(f"Jitter Score     : {jitter_score:.2f}/100")

print()
print("=" * 62)

print(f"FINAL QUALITY SCORE : {quality_score:.2f}/100")
print(f"QUALITY LEVEL       : {quality_level}")

print("=" * 62)

print()
print("Project-defined score weights:")
print("Throughput  : 40%")
print("RTT         : 30%")
print("Packet Loss : 20%")
print("Jitter      : 10%")

print()
print("Task 2 completed successfully.")

Automatic Site-to-Site Path Quality Score
Notebook 2: remote-SEAT -> central-TACC

Historical Average Metrics
--------------------------
Throughput       : 324.165 Mbps
RTT              : 93.334 ms
Packet Loss      : 0.006%
Jitter           : 0.270 ms

Normalized Metric Scores
------------------------
Throughput Score : 64.83/100
RTT Score        : 53.33/100
Loss Score       : 99.87/100
Jitter Score     : 99.73/100

FINAL QUALITY SCORE : 71.88/100
QUALITY LEVEL       : Moderate

Project-defined score weights:
Throughput  : 40%
RTT         : 30%
Packet Loss : 20%
Jitter      : 10%

Task 2 completed successfully.


In [9]:
# ============================================================
# CONTRIBUTION TASK 3
# AUTOMATIC DEGRADED-PATH DETECTION
#
# Notebook 2: remote-SEAT -> central-TACC
#
# Uses the complete OpenSearch archive.
#
# Archive-derived rules:
#   Low Throughput = mean - 2 standard deviations
#   High RTT       = mean + 2 standard deviations
#   High Jitter    = mean + 2 standard deviations
#   Packet Loss    = any value > 0
#
# Thresholds are derived automatically from historical data.
# They are project-defined analytical rules and are not
# official perfSONAR thresholds.
# ============================================================


# ------------------------------------------------------------
# Helper: query OpenSearch
#
# Reuses query_opensearch() created in Task 1.
# ------------------------------------------------------------


# ============================================================
# 1. THROUGHPUT BASELINE STATISTICS
# ============================================================

throughput_stats_query = {
    "size": 0,

    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "throughput"}},
                {"term": {"result.succeeded": True}}
            ]
        }
    },

    "aggs": {
        "stats": {
            "extended_stats": {
                "field": "result.throughput"
            }
        }
    }
}

throughput_stats_data = query_opensearch(
    throughput_stats_query
)

throughput_stats = (
    throughput_stats_data["aggregations"]["stats"]
)

throughput_mean_bps = throughput_stats["avg"]
throughput_std_bps = throughput_stats["std_deviation"]

throughput_threshold_bps = (
    throughput_mean_bps
    - 2 * throughput_std_bps
)

throughput_threshold_mbps = (
    throughput_threshold_bps / 1_000_000
)


# ============================================================
# 2. RTT BASELINE STATISTICS
# ============================================================

rtt_stats_query = {
    "size": 0,

    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "rtt"}},
                {"term": {"result.succeeded": True}}
            ]
        }
    },

    "aggs": {
        "stats": {
            "extended_stats": {
                "field": "result.rtt.mean"
            }
        }
    }
}

rtt_stats_data = query_opensearch(
    rtt_stats_query
)

rtt_stats = (
    rtt_stats_data["aggregations"]["stats"]
)

rtt_mean_seconds = rtt_stats["avg"]
rtt_std_seconds = rtt_stats["std_deviation"]

rtt_threshold_seconds = (
    rtt_mean_seconds
    + 2 * rtt_std_seconds
)

rtt_threshold_ms = (
    rtt_threshold_seconds * 1000
)


# ============================================================
# 3. JITTER BASELINE STATISTICS
# ============================================================

jitter_stats_query = {
    "size": 0,

    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "latencybg"}},
                {"term": {"result.succeeded": True}}
            ]
        }
    },

    "aggs": {
        "stats": {
            "extended_stats": {
                "field": "result.latency.stddev"
            }
        }
    }
}

jitter_stats_data = query_opensearch(
    jitter_stats_query
)

jitter_stats = (
    jitter_stats_data["aggregations"]["stats"]
)

jitter_mean = jitter_stats["avg"]
jitter_std = jitter_stats["std_deviation"]

jitter_threshold = (
    jitter_mean
    + 2 * jitter_std
)


# ============================================================
# 4. COUNT LOW-THROUGHPUT OBSERVATIONS
# ============================================================

low_throughput_query = {
    "size": 0,
    "track_total_hits": True,

    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "throughput"}},
                {"term": {"result.succeeded": True}},

                {
                    "range": {
                        "result.throughput": {
                            "lt": throughput_threshold_bps
                        }
                    }
                }
            ]
        }
    }
}

low_throughput_data = query_opensearch(
    low_throughput_query
)

low_throughput_count = (
    low_throughput_data["hits"]["total"]["value"]
)


# ============================================================
# 5. COUNT HIGH-RTT OBSERVATIONS
# ============================================================

high_rtt_query = {
    "size": 0,
    "track_total_hits": True,

    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "rtt"}},
                {"term": {"result.succeeded": True}},

                {
                    "range": {
                        "result.rtt.mean": {
                            "gt": rtt_threshold_seconds
                        }
                    }
                }
            ]
        }
    }
}

high_rtt_data = query_opensearch(
    high_rtt_query
)

high_rtt_count = (
    high_rtt_data["hits"]["total"]["value"]
)


# ============================================================
# 6. COUNT HIGH-JITTER OBSERVATIONS
# ============================================================

high_jitter_query = {
    "size": 0,
    "track_total_hits": True,

    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "latencybg"}},
                {"term": {"result.succeeded": True}},

                {
                    "range": {
                        "result.latency.stddev": {
                            "gt": jitter_threshold
                        }
                    }
                }
            ]
        }
    }
}

high_jitter_data = query_opensearch(
    high_jitter_query
)

high_jitter_count = (
    high_jitter_data["hits"]["total"]["value"]
)


# ============================================================
# 7. COUNT PACKET-LOSS OBSERVATIONS
# ============================================================

packet_loss_query = {
    "size": 0,
    "track_total_hits": True,

    "query": {
        "bool": {
            "must": [
                {"match": {"test.type": "latencybg"}},
                {"term": {"result.succeeded": True}},

                {
                    "range": {
                        "result.packets.loss": {
                            "gt": 0
                        }
                    }
                }
            ]
        }
    }
}

packet_loss_data = query_opensearch(
    packet_loss_query
)

packet_loss_count = (
    packet_loss_data["hits"]["total"]["value"]
)


# ============================================================
# 8. CALCULATE ABNORMAL-OBSERVATION RATES
# ============================================================

throughput_degradation_rate = (
    low_throughput_count
    / throughput_count * 100
    if throughput_count > 0
    else 0
)

rtt_degradation_rate = (
    high_rtt_count
    / rtt_count * 100
    if rtt_count > 0
    else 0
)

jitter_degradation_rate = (
    high_jitter_count
    / latency_count * 100
    if latency_count > 0
    else 0
)

loss_observation_rate = (
    packet_loss_count
    / latency_count * 100
    if latency_count > 0
    else 0
)


# ============================================================
# 9. AUTOMATIC PATH STATUS
# ============================================================

detected_conditions = []

if low_throughput_count > 0:
    detected_conditions.append(
        "Low-throughput observations"
    )

if high_rtt_count > 0:
    detected_conditions.append(
        "High-RTT observations"
    )

if high_jitter_count > 0:
    detected_conditions.append(
        "High-jitter observations"
    )

if packet_loss_count > 0:
    detected_conditions.append(
        "Packet-loss observations"
    )


if len(detected_conditions) == 0:
    path_status = "NO DEGRADATION DETECTED"

else:
    path_status = "DEGRADATION EVENTS DETECTED"


# ============================================================
# 10. DISPLAY ARCHIVE-DERIVED THRESHOLDS
# ============================================================

print()
print("=" * 72)
print("         ARCHIVE-DERIVED DEGRADATION THRESHOLDS")
print("         Notebook 2: remote-SEAT -> central-TACC")
print("=" * 72)

print()

print(
    f"Throughput historical mean : "
    f"{throughput_mean_bps / 1_000_000:.3f} Mbps"
)

print(
    f"Throughput standard dev.   : "
    f"{throughput_std_bps / 1_000_000:.3f} Mbps"
)

print(
    f"Low throughput threshold   : "
    f"< {throughput_threshold_mbps:.3f} Mbps"
)

print()

print(
    f"RTT historical mean        : "
    f"{rtt_mean_seconds * 1000:.3f} ms"
)

print(
    f"RTT standard deviation     : "
    f"{rtt_std_seconds * 1000:.3f} ms"
)

print(
    f"High RTT threshold         : "
    f"> {rtt_threshold_ms:.3f} ms"
)

print()

print(
    f"Jitter historical mean     : "
    f"{jitter_mean:.3f} ms"
)

print(
    f"Jitter standard deviation  : "
    f"{jitter_std:.3f} ms"
)

print(
    f"High jitter threshold      : "
    f"> {jitter_threshold:.3f} ms"
)

print()

print(
    "Packet-loss threshold      : > 0"
)


# ============================================================
# 11. DISPLAY DETECTED ABNORMAL OBSERVATIONS
# ============================================================

print()
print("=" * 72)
print("             AUTOMATIC DEGRADED-PATH DETECTION")
print("             Notebook 2: remote-SEAT -> central-TACC")
print("=" * 72)

print()

print(
    f"Low-throughput observations : "
    f"{low_throughput_count} / {throughput_count} "
    f"({throughput_degradation_rate:.3f}%)"
)

print(
    f"High-RTT observations       : "
    f"{high_rtt_count} / {rtt_count} "
    f"({rtt_degradation_rate:.3f}%)"
)

print(
    f"High-jitter observations    : "
    f"{high_jitter_count} / {latency_count} "
    f"({jitter_degradation_rate:.3f}%)"
)

print(
    f"Packet-loss observations    : "
    f"{packet_loss_count} / {latency_count} "
    f"({loss_observation_rate:.3f}%)"
)


# ============================================================
# 12. FINAL DETECTOR RESULT
# ============================================================

print()
print("=" * 72)
print("PATH STATUS")
print("=" * 72)

print()
print(path_status)

if detected_conditions:

    print()
    print("Detected conditions:")

    for condition in detected_conditions:
        print(f"  - {condition}")

else:

    print()
    print(
        "No abnormal conditions were detected "
        "using the archive-derived rules."
    )


print()
print(
    "Note: thresholds are derived from the historical archive "
    "using mean +/- 2 standard deviations."
)

print(
    "These are project-defined analytical rules and are not "
    "official perfSONAR thresholds."
)

print()
print(
    "Task 3: Automatic degraded-path detection completed."
)

{"took":26,"timed_out":false,"_shards":{"total":60,"successful":60,"skipped":0,"failed":0},"hits":{"total":{"value":7042,"relation":"eq"},"max_score":null,"hits":[]},"aggregations":{"stats":{"count":7042,"min":3820547.0,"max":3.510624E8,"avg":3.2416532936474013E8,"sum":2.2827722493865E12,"sum_of_squares":7.43279914225308E20,"variance":4.663868415570135E14,"variance_population":4.663868415570135E14,"variance_sampling":4.664530802790071E14,"std_deviation":2.1595991330731116E7,"std_deviation_population":2.1595991330731116E7,"std_deviation_sampling":2.1597524864646114E7,"std_deviation_bounds":{"upper":3.673573120262024E8,"lower":2.809733467032779E8,"upper_population":3.673573120262024E8,"lower_population":2.809733467032779E8,"upper_sampling":3.6736037909403235E8,"lower_sampling":2.809702796354479E8}}}}{"took":21,"timed_out":false,"_shards":{"total":60,"successful":60,"skipped":0,"failed":0},"hits":{"total":{"value":8044,"relation":"eq"},"max_score":null,"hits":[]},"aggregations":{"stats":{

In [10]:
# ============================================================
# CONTRIBUTION TASK 4
# AUTOMATED EXPERIMENT SUMMARY
#
# Notebook 2: remote-SEAT -> central-TACC
#
# Combines:
#   Task 1 -> Full-archive performance analysis
#   Task 2 -> Site quality score
#   Task 3 -> Automatic degradation detection
#
# All reported values are derived from the archive.
# ============================================================


# ------------------------------------------------------------
# 1. Collect degradation results from Task 3
# ------------------------------------------------------------

degradation_results = {
    "Low throughput": {
        "count": low_throughput_count,
        "total": throughput_count,
        "rate": throughput_degradation_rate
    },

    "High RTT": {
        "count": high_rtt_count,
        "total": rtt_count,
        "rate": rtt_degradation_rate
    },

    "High jitter": {
        "count": high_jitter_count,
        "total": latency_count,
        "rate": jitter_degradation_rate
    },

    "Packet loss": {
        "count": packet_loss_count,
        "total": latency_count,
        "rate": loss_observation_rate
    }
}


# ------------------------------------------------------------
# 2. Identify most frequent abnormal condition by rate
# ------------------------------------------------------------

dominant_condition = max(
    degradation_results,
    key=lambda key: degradation_results[key]["rate"]
)

dominant_rate = (
    degradation_results[
        dominant_condition
    ]["rate"]
)


# ------------------------------------------------------------
# 3. Count categories containing degradation observations
# ------------------------------------------------------------

active_conditions = [
    condition
    for condition, result
    in degradation_results.items()
    if result["count"] > 0
]


# ------------------------------------------------------------
# 4. Determine experiment-level status
#
# This is descriptive only.
# We report whether abnormal observations were present,
# rather than assigning an arbitrary STABLE/VARIABLE label.
# ------------------------------------------------------------

if len(active_conditions) == 0:

    experiment_status = (
        "NO DEGRADATION OBSERVATIONS DETECTED"
    )

else:

    experiment_status = (
        "DEGRADATION OBSERVATIONS DETECTED"
    )


# ------------------------------------------------------------
# 5. Display experiment summary
# ------------------------------------------------------------

print()
print("=" * 76)
print("                  AUTOMATED EXPERIMENT SUMMARY")
print("                  Notebook 2: remote-SEAT -> central-TACC")
print("=" * 76)

print()

print("FULL-ARCHIVE PERFORMANCE")
print("-" * 76)

print(
    f"Throughput measurements : "
    f"{throughput_count}"
)

print(
    f"Average throughput      : "
    f"{throughput_avg:.3f} Mbps"
)

print(
    f"Minimum throughput      : "
    f"{throughput_min:.3f} Mbps"
)

print(
    f"Maximum throughput      : "
    f"{throughput_max:.3f} Mbps"
)

print()

print(
    f"RTT measurements        : "
    f"{rtt_count}"
)

print(
    f"Average RTT             : "
    f"{rtt_avg:.3f} ms"
)

print(
    f"Minimum RTT             : "
    f"{rtt_min:.3f} ms"
)

print(
    f"Maximum RTT             : "
    f"{rtt_max:.3f} ms"
)

print(
    f"RTT packet loss         : "
    f"{rtt_loss:.3f}%"
)

print()

print(
    f"Latency measurements    : "
    f"{latency_count}"
)

print(
    f"Average latency         : "
    f"{latency_avg:.3f} ms"
)

print(
    f"Average jitter          : "
    f"{jitter_avg:.3f} ms"
)

print(
    f"Latency packet loss     : "
    f"{latency_loss:.3f}%"
)


# ------------------------------------------------------------
# 6. Display quality-score result
# ------------------------------------------------------------

print()
print("=" * 76)
print("PATH QUALITY")
print("=" * 76)

print()

print(
    f"Quality score : "
    f"{quality_score:.2f}/100"
)

print(
    f"Quality level : "
    f"{quality_level}"
)

print()

print("Metric scores:")

print(
    f"  Throughput : "
    f"{throughput_score:.2f}/100"
)

print(
    f"  RTT        : "
    f"{rtt_score:.2f}/100"
)

print(
    f"  Loss       : "
    f"{loss_score:.2f}/100"
)

print(
    f"  Jitter     : "
    f"{jitter_score:.2f}/100"
)


# ------------------------------------------------------------
# 7. Display degradation analysis
# ------------------------------------------------------------

print()
print("=" * 76)
print("AUTOMATIC DEGRADATION ANALYSIS")
print("=" * 76)

print()

for condition, result in degradation_results.items():

    print(
        f"{condition:<16}: "
        f"{result['count']:>7} / "
        f"{result['total']:<7} "
        f"({result['rate']:.3f}%)"
    )


# ------------------------------------------------------------
# 8. Display dominant abnormal condition
# ------------------------------------------------------------

print()

if active_conditions:

    print(
        f"Highest abnormal-observation rate : "
        f"{dominant_condition}"
    )

    print(
        f"Highest abnormal-observation rate : "
        f"{dominant_rate:.3f}%"
    )

else:

    print(
        "No abnormal observation category "
        "contained detected events."
    )


# ------------------------------------------------------------
# 9. Overall experiment interpretation
# ------------------------------------------------------------

print()
print("=" * 76)
print("OVERALL EXPERIMENT INTERPRETATION")
print("=" * 76)

print()

print(
    f"Experiment status : "
    f"{experiment_status}"
)

print(
    f"Archive-derived quality level : "
    f"{quality_level}"
)

print()

if active_conditions:

    print(
        "The archive contains measurable abnormal observations, "
        "but each detected condition occurs in a relatively small "
        "fraction of its corresponding measurements."
    )

    print(
        f"The largest detected abnormal-observation rate is "
        f"{dominant_rate:.3f}% for {dominant_condition.lower()}."
    )

else:

    print(
        "No degradation observations were detected using "
        "the archive-derived rules."
    )


# ------------------------------------------------------------
# 10. Explain overall score
# ------------------------------------------------------------

print()

if throughput_score < rtt_score:

    limiting_metric = "throughput"

else:

    limiting_metric = "RTT"

print(
    f"The overall quality score is primarily limited by "
    f"{limiting_metric}, while packet-loss and jitter scores "
    f"remain close to 100."
)


# ------------------------------------------------------------
# 11. Methodological note
# ------------------------------------------------------------

print()

print(
    "Note: quality scores and degradation thresholds are "
    "project-defined analytical methods and are not official "
    "perfSONAR metrics or classifications."
)

print()

print(
    "Task 4: Automated experiment summary completed."
)


                  AUTOMATED EXPERIMENT SUMMARY
                  Notebook 2: remote-SEAT -> central-TACC

FULL-ARCHIVE PERFORMANCE
----------------------------------------------------------------------------
Throughput measurements : 7042
Average throughput      : 324.165 Mbps
Minimum throughput      : 3.821 Mbps
Maximum throughput      : 351.062 Mbps

RTT measurements        : 8044
Average RTT             : 93.334 ms
Minimum RTT             : 92.397 ms
Maximum RTT             : 234.248 ms
RTT packet loss         : 0.007%

Latency measurements    : 2218616
Average latency         : 42.683 ms
Average jitter          : 0.270 ms
Latency packet loss     : 0.006%

PATH QUALITY

Quality score : 71.88/100
Quality level : Moderate

Metric scores:
  Throughput : 64.83/100
  RTT        : 53.33/100
  Loss       : 99.87/100
  Jitter     : 99.73/100

AUTOMATIC DEGRADATION ANALYSIS

Low throughput  :     116 / 7042    (1.647%)
High RTT        :      60 / 8044    (0.746%)
High jitter     :    3020 /